In [ ]:
# ============================================================
# CELL 1 — Install and import libraries
# ============================================================

!pip -q install timm scikit-learn tqdm

import os
import zipfile
import random
import time
import copy
import warnings
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

print("✅ Imports successful")
print("PyTorch version:", torch.__version__)
print("timm version:", timm.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# CELL 2 — Configuration
# ============================================================

SEED = 42
ZIP_PATH = "/content/archive (5).zip"
EXTRACT_PATH = "/content/extracted_data"
MODEL_SAVE_DIR = "/content/working_transformers"

IMAGE_SIZE = 224
BATCH_SIZE = 16          # For Colab T4. If GPU memory error occurs, set 8.
NUM_WORKERS = 2
VALIDATION_SPLIT_RATIO = 0.20

PHASE1_EPOCHS = 10       # Train classifier head only
PHASE2_EPOCHS = 20       # Fine-tune full Transformer
PHASE1_LR = 1e-3
PHASE2_LR = 1e-5
WEIGHT_DECAY = 1e-4
PATIENCE_PHASE1 = 5
PATIENCE_PHASE2 = 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("✅ Configuration complete")
print("Device:", DEVICE)


In [ ]:
# ============================================================
# CELL 3 — Extract dataset and locate train/test folders
# ============================================================

os.makedirs(EXTRACT_PATH, exist_ok=True)

if os.path.exists(ZIP_PATH):
    print(f"Extracting dataset from: {ZIP_PATH}")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("✅ Dataset extracted to:", EXTRACT_PATH)
else:
    print(f"❌ ZIP file not found at {ZIP_PATH}")
    print("Please upload your dataset ZIP to Colab or update ZIP_PATH.")

# Original paths from your CNN notebook
train_dir = f"{EXTRACT_PATH}/PMRAM Bangladeshi Brain Cancer - MRI Dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/Augmented Data/Augmented"
test_dir = f"{EXTRACT_PATH}/PMRAM Bangladeshi Brain Cancer - MRI Dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/Raw Data/Raw"

# Fallback auto-finder if folder names are nested differently
def find_directory_by_suffix(root, suffix_parts):
    suffix = os.path.join(*suffix_parts)
    matches = []
    for current_root, dirs, files in os.walk(root):
        normalized = current_root.replace("\\", "/")
        target = suffix.replace("\\", "/")
        if normalized.endswith(target):
            matches.append(current_root)
    return matches[0] if matches else None

if not os.path.exists(train_dir):
    detected_train = find_directory_by_suffix(EXTRACT_PATH, ["Augmented Data", "Augmented"])
    if detected_train:
        train_dir = detected_train

if not os.path.exists(test_dir):
    detected_test = find_directory_by_suffix(EXTRACT_PATH, ["Raw Data", "Raw"])
    if detected_test:
        test_dir = detected_test

print("Train directory:", train_dir)
print("Test directory:", test_dir)

if not os.path.exists(train_dir):
    raise FileNotFoundError("Training directory not found. Check ZIP_PATH/extraction structure.")
if not os.path.exists(test_dir):
    raise FileNotFoundError("Testing directory not found. Check ZIP_PATH/extraction structure.")

print("\n--- Training Data Count ---")
for category in sorted(os.listdir(train_dir)):
    folder = os.path.join(train_dir, category)
    if os.path.isdir(folder):
        print(f"{category}: {len(os.listdir(folder))} images")

print("\n--- Testing Data Count ---")
for category in sorted(os.listdir(test_dir)):
    folder = os.path.join(test_dir, category)
    if os.path.isdir(folder):
        print(f"{category}: {len(os.listdir(folder))} images")


In [ ]:
# ============================================================
# CELL 4 — Show sample MRI images
# ============================================================

categories = [c for c in sorted(os.listdir(train_dir)) if os.path.isdir(os.path.join(train_dir, c))]

plt.figure(figsize=(10, 8))

for idx, category in enumerate(categories[:4]):
    folder = os.path.join(train_dir, category)
    image_files = [f for f in os.listdir(folder) if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))]
    if not image_files:
        continue

    img_path = os.path.join(folder, image_files[min(1, len(image_files)-1)])
    img = Image.open(img_path).convert("RGB")

    plt.subplot(2, 2, idx + 1)
    plt.imshow(img)
    plt.title(category)
    plt.axis("off")

plt.tight_layout()
plt.suptitle("Sample Augmented MRI Images", fontsize=16, y=1.02)
plt.show()


In [ ]:
# ============================================================
# CELL 5 — Create PyTorch datasets and dataloaders
# ============================================================

# ImageNet normalization is used because ViT and Swin are pretrained on ImageNet-style RGB images.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08), shear=8),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Dataset only for index/label reading
split_reference_dataset = datasets.ImageFolder(train_dir)
class_names = split_reference_dataset.classes
NUM_CLASSES = len(class_names)

targets = np.array([label for _, label in split_reference_dataset.samples])
indices = np.arange(len(targets))

train_indices, val_indices = train_test_split(
    indices,
    test_size=VALIDATION_SPLIT_RATIO,
    random_state=SEED,
    stratify=targets
)

# Separate datasets so train and validation can use different transforms
train_full_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_full_dataset = datasets.ImageFolder(train_dir, transform=eval_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)

# Safety check: class order should match between train and test folders
print("Train class_to_idx:", train_full_dataset.class_to_idx)
print("Test class_to_idx :", test_dataset.class_to_idx)

if train_full_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("Train and test class folder names/order do not match. Please check dataset folders.")

train_dataset = Subset(train_full_dataset, train_indices)
val_dataset = Subset(val_full_dataset, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("\n✅ DataLoaders created successfully")
print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))


In [ ]:
# ============================================================
# CELL 6 — Utility functions: AMP, plots, evaluation
# ============================================================

def get_autocast_context():
    """Works with current PyTorch AMP API and older CUDA AMP fallback."""
    if DEVICE.type != "cuda":
        return nullcontext()

    try:
        return torch.amp.autocast("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.autocast(enabled=True)

def create_grad_scaler():
    if DEVICE.type != "cuda":
        return None

    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)

def plot_training_history(history_df, model_name):
    plt.figure(figsize=(16, 6))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history_df["epoch_global"], history_df["train_acc"], label="Train Accuracy")
    plt.plot(history_df["epoch_global"], history_df["val_acc"], label="Val Accuracy")

    if "phase" in history_df.columns:
        phase1_rows = history_df[history_df["phase"] == "Phase 1"]
        if len(phase1_rows) > 0:
            phase1_end = phase1_rows["epoch_global"].max()
            plt.axvline(x=phase1_end, linestyle="--", color="gray", label="End Phase 1")

    plt.ylim([0, 1.05])
    plt.legend(loc="lower right")
    plt.title(f"{model_name} - Accuracy per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True, linestyle="--", alpha=0.7)

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history_df["epoch_global"], history_df["train_loss"], label="Train Loss")
    plt.plot(history_df["epoch_global"], history_df["val_loss"], label="Val Loss")

    if "phase" in history_df.columns:
        phase1_rows = history_df[history_df["phase"] == "Phase 1"]
        if len(phase1_rows) > 0:
            phase1_end = phase1_rows["epoch_global"].max()
            plt.axvline(x=phase1_end, linestyle="--", color="gray", label="End Phase 1")

    plt.legend(loc="upper right")
    plt.title(f"{model_name} - Loss per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()

def evaluate_model(model, dataloader, class_names, model_name="Model"):
    model.eval()

    all_labels = []
    all_probs = []
    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with get_autocast_context():
                outputs = model(images)
                loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = np.argmax(y_prob, axis=1)

    avg_loss = total_loss / max(total_samples, 1)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print(f"\n{'='*60}")
    print(f"📊 Evaluation Results for {model_name}")
    print(f"{'='*60}")
    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Weighted Precision: {precision:.4f}")
    print(f"Weighted Recall: {recall:.4f}")
    print(f"Weighted F1-score: {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=1,
        linecolor="black"
    )
    plt.title(f"{model_name} - Confusion Matrix", fontsize=14, pad=15)
    plt.xlabel("Predicted Labels", fontsize=12, labelpad=10)
    plt.ylabel("True Labels", fontsize=12, labelpad=10)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    auc = None
    try:
        y_true_one_hot = label_binarize(y_true, classes=list(range(len(class_names))))
        auc = roc_auc_score(y_true_one_hot, y_prob, multi_class="ovr")
        print(f"📈 AUC Score: {auc:.4f}")
    except Exception as e:
        print(f"⚠️ AUC calculation failed: {e}")

    return {
        "model": model_name,
        "test_loss": avg_loss,
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "auc_ovr": auc
    }


In [ ]:
# ============================================================
# CELL 7 — Build and freeze/unfreeze pretrained Transformer models
# ============================================================

def build_timm_model(timm_model_name, num_classes):
    """
    Creates a pretrained ViT/Swin model and replaces the classifier head
    for the MRI dataset classes.
    """
    model = timm.create_model(
        timm_model_name,
        pretrained=True,
        num_classes=num_classes
    )
    return model

def freeze_all_layers(model):
    for param in model.parameters():
        param.requires_grad = False

def unfreeze_all_layers(model):
    for param in model.parameters():
        param.requires_grad = True

def set_head_trainable_only(model):
    """
    Phase 1: freeze backbone and train classifier head only.
    timm models usually expose the classifier through get_classifier().
    """
    freeze_all_layers(model)

    classifier = model.get_classifier()

    if isinstance(classifier, nn.Module):
        for param in classifier.parameters():
            param.requires_grad = True
    else:
        # Fallback for uncommon classifier structures
        for name, param in model.named_parameters():
            if any(key in name.lower() for key in ["head", "classifier", "fc"]):
                param.requires_grad = True

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def print_trainable_summary(model, model_name):
    trainable, total = count_trainable_params(model)
    print(f"{model_name}: trainable parameters = {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


In [ ]:
# ============================================================
# CELL 8 — Training functions: two-phase Transformer fine-tuning
# ============================================================

def run_one_epoch(model, dataloader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    if is_train:
        context = nullcontext()
    else:
        context = torch.no_grad()

    with context:
        for images, labels in dataloader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with get_autocast_context():
                outputs = model(images)
                loss = criterion(outputs, labels)

            if is_train:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            _, preds = torch.max(outputs, dim=1)
            running_correct += torch.sum(preds == labels).item()
            total_samples += batch_size

    epoch_loss = running_loss / max(total_samples, 1)
    epoch_acc = running_correct / max(total_samples, 1)
    return epoch_loss, epoch_acc

def fit_phase(
    model,
    phase_name,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs,
    patience,
    save_path,
    starting_global_epoch=0
):
    scaler = create_grad_scaler()
    best_val_acc = -1.0
    best_epoch = 0
    patience_counter = 0
    phase_history = []

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        train_loss, train_acc = run_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            scaler=scaler
        )

        val_loss, val_acc = run_one_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None,
            scaler=None
        )

        if scheduler is not None:
            scheduler.step(val_acc)

        current_lr = optimizer.param_groups[0]["lr"]
        epoch_global = starting_global_epoch + epoch

        row = {
            "phase": phase_name,
            "epoch_in_phase": epoch,
            "epoch_global": epoch_global,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": current_lr,
            "time_sec": time.time() - epoch_start
        }
        phase_history.append(row)

        print(
            f"{phase_name} | Epoch {epoch:02d}/{epochs:02d} "
            f"| Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} "
            f"| LR: {current_lr:.2e}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved best weights: {save_path}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"⏹️ Early stopping in {phase_name}. Best epoch: {best_epoch}, Best Val Acc: {best_val_acc:.4f}")
                break

    return phase_history

def train_two_phase_transformer(display_name, timm_model_name):
    print(f"\n{'='*70}")
    print(f"🚀 Training {display_name} ({timm_model_name})")
    print(f"{'='*70}")

    phase1_path = os.path.join(MODEL_SAVE_DIR, f"{display_name.lower()}_phase1_best.pth")
    phase2_path = os.path.join(MODEL_SAVE_DIR, f"{display_name.lower()}_finetuned_best.pth")

    model = build_timm_model(timm_model_name, NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # -----------------------------
    # Phase 1: Train classifier head
    # -----------------------------
    print("\n--- Phase 1: Training classifier head only ---")
    set_head_trainable_only(model)
    print_trainable_summary(model, display_name)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=PHASE1_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    history_p1 = fit_phase(
        model=model,
        phase_name="Phase 1",
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        epochs=PHASE1_EPOCHS,
        patience=PATIENCE_PHASE1,
        save_path=phase1_path,
        starting_global_epoch=0
    )

    if os.path.exists(phase1_path):
        model.load_state_dict(torch.load(phase1_path, map_location=DEVICE))
        print("✅ Loaded best Phase 1 weights")

    # -----------------------------
    # Phase 2: Fine-tune full model
    # -----------------------------
    print("\n--- Phase 2: Fine-tuning full Transformer ---")
    unfreeze_all_layers(model)
    print_trainable_summary(model, display_name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE2_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    history_p2 = fit_phase(
        model=model,
        phase_name="Phase 2",
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        epochs=PHASE2_EPOCHS,
        patience=PATIENCE_PHASE2,
        save_path=phase2_path,
        starting_global_epoch=len(history_p1)
    )

    if os.path.exists(phase2_path):
        model.load_state_dict(torch.load(phase2_path, map_location=DEVICE))
        print("✅ Loaded best Phase 2 fine-tuned weights")

    history = pd.DataFrame(history_p1 + history_p2)
    history_csv_path = os.path.join(MODEL_SAVE_DIR, f"{display_name.lower()}_history.csv")
    history.to_csv(history_csv_path, index=False)

    print(f"✅ Training history saved to: {history_csv_path}")
    print(f"✅ Best fine-tuned model saved to: {phase2_path}")

    return model, history, phase2_path


In [ ]:
# ============================================================
# CELL 9 — Train ViT and Swin models
# ============================================================

MODEL_CONFIGS = [
    {
        "display_name": "ViT_Base",
        "timm_name": "vit_base_patch16_224"
    },
    {
        "display_name": "Swin_Tiny",
        "timm_name": "swin_tiny_patch4_window7_224"
    },

    # Optional heavier Swin model.
    # Use only if you have a stronger GPU/high RAM.
    # {
    #     "display_name": "Swin_Base",
    #     "timm_name": "swin_base_patch4_window7_224"
    # },
]

trained_models = {}
histories = {}
model_paths = {}

for cfg in MODEL_CONFIGS:
    display_name = cfg["display_name"]
    timm_name = cfg["timm_name"]

    model, history, best_path = train_two_phase_transformer(display_name, timm_name)

    trained_models[display_name] = model
    histories[display_name] = history
    model_paths[display_name] = best_path

    plot_training_history(history, display_name)

    # Clear unused GPU memory before the next model
    torch.cuda.empty_cache()

print("\n🎉 ViT and Swin training completed.")
print("Saved model paths:")
for name, path in model_paths.items():
    print(f"{name}: {path}")


In [ ]:
# ============================================================
# CELL 10 — Final evaluation on Raw Test Data
# ============================================================

print("==================================================")
print("             STARTING FINAL EVALUATION            ")
print("==================================================")

results = []

for cfg in MODEL_CONFIGS:
    display_name = cfg["display_name"]
    timm_name = cfg["timm_name"]
    best_path = model_paths.get(display_name)

    print(f"\n{'*'*70}")
    print(f"🔍 Processing {display_name} Evaluation")
    print(f"{'*'*70}")

    if best_path is None or not os.path.exists(best_path):
        print(f"❌ Best model path not found for {display_name}")
        continue

    model = build_timm_model(timm_name, NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f"✅ Loaded model from: {best_path}")

    metrics = evaluate_model(
        model=model,
        dataloader=test_loader,
        class_names=class_names,
        model_name=display_name
    )

    metrics["checkpoint"] = best_path
    results.append(metrics)

    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_csv = os.path.join(MODEL_SAVE_DIR, "vit_swin_final_test_results.csv")
results_df.to_csv(results_csv, index=False)

print("\n✅ Final comparison table:")
display(results_df)

print(f"\n✅ Results saved to: {results_csv}")
print("\n🎉 Evaluation pipeline completed successfully!")


In [ ]:
# ============================================================
# CELL 11 — Optional: Single image prediction
# ============================================================

def predict_single_image(image_path, timm_model_name, checkpoint_path, class_names):
    model = build_timm_model(timm_model_name, len(class_names)).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()

    img = Image.open(image_path).convert("RGB")
    tensor = eval_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        with get_autocast_context():
            output = model(tensor)
            prob = torch.softmax(output, dim=1).detach().cpu().numpy()[0]

    pred_idx = int(np.argmax(prob))
    pred_class = class_names[pred_idx]
    confidence = float(prob[pred_idx])

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Prediction: {pred_class} | Confidence: {confidence:.4f}")
    plt.show()

    return {
        "predicted_class": pred_class,
        "confidence": confidence,
        "all_probabilities": {class_names[i]: float(prob[i]) for i in range(len(class_names))}
    }

# Example usage:
# image_path = "/content/example_mri.jpg"
# output = predict_single_image(
#     image_path=image_path,
#     timm_model_name="vit_base_patch16_224",
#     checkpoint_path=model_paths["ViT_Base"],
#     class_names=class_names
# )
# print(output)


## Notes

1. If Colab gives GPU memory error, reduce `BATCH_SIZE` from `16` to `8` or `4`.
2. `ViT_Base` is heavier than `Swin_Tiny`; Swin Tiny usually runs faster on Colab.
3. The code uses a two-phase pipeline:
   - Phase 1: freeze pretrained backbone and train only the classifier head.
   - Phase 2: unfreeze the full Transformer and fine-tune using a small learning rate.
4. Final evaluation includes:
   - Accuracy
   - Weighted precision
   - Weighted recall
   - Weighted F1-score
   - Confusion matrix
   - Multi-class AUC-OVR, when computable
